# Linear Regression Implementation from Scratch

This notebook implements a linear regression model using gradient descent to predict house prices based on various features.

## Mathematical Background

### Hypothesis Function
The model predicts the output $y$ using a linear combination of input features $x$:
$$ h_w(x) = w \cdot x + b $$
Where:
- $w$ is the weight vector
- $b$ is the bias term

### Cost Function (Mean Squared Error)
We measure the error using the Mean Squared Error (MSE):
$$ J(w, b) = \frac{1}{2m} \sum_{i=1}^{m} (h_w(x^{(i)}) - y^{(i)})^2 $$
Where $m$ is the number of training examples.

### Gradient Descent
To minimize the cost function, we update the parameters iteratively:

**Update Rules:**
$$ w_j := w_j - \alpha \frac{\partial J}{\partial w_j} $$
$$ b := b - \alpha \frac{\partial J}{\partial b} $$

**Gradients:**
$$ \frac{\partial J}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} (h_w(x^{(i)}) - y^{(i)}) x_j^{(i)} $$
$$ \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (h_w(x^{(i)}) - y^{(i)}) $$

Where $\alpha$ is the learning rate.

## 1. Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import copy

## 2. Data Loading and Exploration

We load the housing dataset and perform basic exploration to understand the data structure.

In [ ]:
# Load the dataset
housing_data = pd.read_csv('./Housing.csv')

# Display first few rows
housing_data.head()

In [ ]:
# Basic info about the dataset
housing_data.info()
housing_data.describe(include='all')

In [ ]:
# Check for missing values and duplicates
print("Missing values:", housing_data.isnull().sum().sum())
print("Duplicates:", housing_data.duplicated().sum())

## 3. Data Preprocessing

### 3.1 Converting Categorical Values to Numeric
Machine learning models require numerical input. We convert categorical columns (Yes/No, Furnished/Semi-furnished/etc.) into numeric values.

In [ ]:
# Identify categorical columns
cat_col = housing_data.select_dtypes(include=['object']).columns
print(f"Categorical columns: {list(cat_col)}")

# Convert categorical values to numeric
for col in cat_col:
    unique_vals = housing_data[col].unique()
    mapping = list(range(len(unique_vals)))
    print(f"{col}: {unique_vals} -> {mapping}")
    housing_data[col] = housing_data[col].replace(unique_vals, mapping)

housing_data.head()

### 3.2 Feature Scaling (Z-Score Normalization)

We normalize features to have a mean of 0 and a standard deviation of 1. This helps gradient descent converge faster.

Formula:
$$ x_{norm} = \frac{x - \mu}{\sigma} $$
Where $\mu$ is the mean and $\sigma$ is the standard deviation.

In [ ]:
def zscore_normalize_features(X):
    """
    Normalizes features using Z-score normalization.
    """
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

# Prepare training data
X_train = np.array(housing_data.drop('price', axis=1).values)
y_train = np.array(housing_data['price'].values)

# Convert to float
X_train = X_train.astype(float)

# Normalize features
X_train, mu, sigma = zscore_normalize_features(X_train)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

## 4. Implementing the Model

### 4.1 Cost Function

Calculates the Mean Squared Error (MSE) cost.

Formula: $J(w, b) = \frac{1}{2m} \sum_{i=1}^{m} (h_w(x^{(i)}) - y^{(i)})^2$

In [ ]:
def compute_cost(X, y, w, b): 
    """
    Computes the cost function for linear regression.
    Args:
      X (ndarray (m, n)): Data, m examples 
      y (ndarray (m,)): target values
      w (ndarray (n,)): model parameters  
      b (scalar): model parameter  
    """
    m = X.shape[0] 
    cost = 0.0
    for i in range(m):                                
        f_wb_i = np.dot(X[i], w) + b           # Hypothesis h(x)
        cost = cost + (f_wb_i - y[i])**2       # Squared error
    cost = cost / (2 * m)                      # Mean squared error
    return cost

### 4.2 Compute Gradient

Calculates the partial derivatives of the cost function with respect to $w$ and $b$.

**Formulas:**
$$ \frac{\partial J}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} (h_w(x^{(i)}) - y^{(i)}) x_j^{(i)} $$
$$ \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (h_w(x^{(i)}) - y^{(i)}) $$

In [ ]:
def compute_gradient(X, y, w, b): 
    """
    Computes the gradient for linear regression 
    """
    m,n = X.shape           # (number of examples, number of features)
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):                             
        err = (np.dot(X[i], w) + b) - y[i]   # Error = prediction - actual
        for j in range(n):                         
            dj_dw[j] = dj_dw[j] + err * X[i, j]    # Derivative w.r.t w
        dj_db = dj_db + err                        # Derivative w.r.t b
        
    dj_dw = dj_dw / m                                
    dj_db = dj_db / m                                
        
    return dj_db, dj_dw

### 4.3 Gradient Descent

Iteratively updates the parameters $w$ and $b$ to minimize the cost function.

**Update Rules:**
$$ w := w - \alpha \frac{\partial J}{\partial w} $$
$$ b := b - \alpha \frac{\partial J}{\partial b} $$

Where $\alpha$ is the learning rate.

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    """
    Performs batch gradient descent to learn w and b.
    """
    J_history = []
    w = copy.deepcopy(w_in)  # Avoid modifying global w
    b = b_in
    
    for i in range(num_iters):
        # Calculate the gradient
        dj_db, dj_dw = gradient_function(X, y, w, b)   

        # Update Parameters
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               
      
        # Save cost J at each iteration
        if i < 100000:
            J_history.append(cost_function(X, y, w, b))

        # Print cost every 10% of iterations
        if i % math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}")
        
    return w, b, J_history

## 5. Training the Model

We initialize weights and bias, then run gradient descent.

In [ ]:
# Initialize parameters
w_init = np.zeros(X_train.shape[1])
b_init = 0

# Hyperparameters
iterations = 1000
alpha = 0.01

# Run gradient descent
w_final, b_final, J_hist = gradient_descent(
    X_train, y_train, w_init, b_init, compute_cost, compute_gradient, alpha, iterations
)

print(f"\n(w,b) found by gradient descent: ({w_final}, {b_final})")

## 6. Visualization and Evaluation

### 6.1 Cost Function Convergence
Plotting the cost $J$ over iterations helps verify that the algorithm is converging.

In [ ]:
# Plot cost versus iteration
plt.figure(figsize=(10, 6))
plt.plot(J_hist)
plt.title("Cost Function J per Iteration")
plt.ylabel('Cost')
plt.xlabel('Iteration Step')
plt.grid(True)
plt.show()

### 6.2 Predictions vs Actual
Visualize how well the predicted prices match the actual prices.

In [ ]:
# Get predictions
y_pred = np.dot(X_train, w_final) + b_final

# Plotting
plt.figure(figsize=(8, 6))
plt.scatter(y_train, y_pred, alpha=0.5, color='teal')

# Perfect prediction line
max_val = max(max(y_train), max(y_pred))
min_val = min(min(y_train), min(y_pred))
plt.plot([min_val, max_val], [min_val, max_val], color='red', lw=2, linestyle='--')

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs. Predicted Housing Prices")
plt.grid(True)
plt.show()

### 6.3 Feature Importance
Visualize the weights ($w$) to see which features have the most impact on the price.

In [ ]:
feature_names = housing_data.drop('price', axis=1).columns
plt.figure(figsize=(10, 6))
plt.barh(feature_names, w_final, color='skyblue')
plt.xlabel("Weight Value (Importance)")
plt.title("Impact of Each Feature on House Price")
plt.show()

### 6.4 Model Accuracy
Calculate Mean Absolute Percentage Error (MAPE).

In [ ]:
# Mean Absolute Percentage Error (MAPE)
mape = np.mean(np.abs((y_train - y_pred) / y_train)) * 100
print(f"Mean Absolute Percentage Error: {mape:.2f}%")
print(f"Model Accuracy (100 - MAPE): {100 - mape:.2f}%")

## 7. Making New Predictions

Function to predict the price of a new house given its raw features.

In [ ]:
def predict_house_price(raw_data, w, b, mu, sigma):
    """
    Takes raw house features, normalizes them, and returns a predicted price.
    
    Args:
      raw_data (list or np.array): The 12 features of the house
      w, b: Your trained model parameters
      mu, sigma: Mean and StdDev from your training set
    """
    # Convert to numpy array
    x_input = np.array(raw_data)
    
    # Normalize using training statistics (Crucial!)
    x_norm = (x_input - mu) / sigma
    
    # Compute prediction: y = wx + b
    prediction = np.dot(x_norm, w) + b
    
    return prediction

In [ ]:
# Example predictions
my_house1 = [7500, 4, 2, 2, 1, 1, 1, 0, 1, 2, 1, 0]
my_house2 = [3000, 2, 1, 1, 0, 0, 0, 0, 0, 0, 0, 2]

price1 = predict_house_price(my_house1, w_final, b_final, mu, sigma)
price2 = predict_house_price(my_house2, w_final, b_final, mu, sigma)

print(f"--- Prediction Result ---")
print(f"Estimated Market Value House1: ${price1:,.2f}")
print(f"Estimated Market Value House2: ${price2:,.2f}")